# Domain 3: Claude Code Configuration & Workflows
## Task Statement 3.5: Iterative Refinement & The Interview Pattern
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to apply the Interview Pattern to resolve underspecified requirements before code generation, use concrete I/O examples, and perform test-driven iterative refinement.

---

## 🏗️ Architecture: The Interview Gatekeeper Pattern

```mermaid
flowchart TD
    UserReq["User Prompt:<br>'Build an authentication service'"] --> Analyzer["Ambiguity & Scope Analyzer"]
    
    Analyzer --> AmbiguityCheck{"Scope Underspecified?"}
    
    AmbiguityCheck -->|"Yes: High Ambiguity"| Interview["Interview Mode Activated"]
    Interview --> Ask["Clarifying Questions (2-3 Options)<br>Auth Type, DB & Session Model"]
    Ask --> UserFeedback["User Selects Exact Architecture"]
    
    UserFeedback --> ConcreteIO["Provide 2-3 Concrete<br>Input/Output Examples"]
    AmbiguityCheck -->|"No: Specified"| ConcreteIO
    
    ConcreteIO --> TestCases["Generate Test Suite<br>First (TDD)"]
    TestCases --> CodeGen["Implement Targeted Solution"]
    CodeGen --> Verification["Run Offline Test Assertions"]
    Verification --> Success["Refinement Complete"]
```

---

## ⚠️ Anti-Patterns vs Best Practices

| Anti-Pattern (Fails on Exam) | Recommended Architectural Practice |
|---|---|
| **Silent Guesswork**: Claude assumes JWT + MongoDB without asking and implements 500 lines of throwaway code. | **The Interview Pattern**: Halt execution, outline 2-3 explicit options, and request user confirmation before writing code. |
| **Fragmented Hotfixes**: Fixing one bug at a time across 5 turns, introducing regression cycles. | **Interacting Issue Analysis**: Identify coupling across modules and resolve interacting defects in a unified plan. |
| **Abstract Specifications**: Describing output format purely in prose without examples. | **Concrete Examples**: Provide 2-3 exact Input/Output pairs with boundary conditions. |

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.5 Iterative Refinement & Interview Pattern.")

### 📋 Step 2: Clarification & Grounded Spec Coordinator
We implement the ambiguity evaluator and specification compiler.

In [ ]:
# Step 2: Refinement Coordinator Implementation
@dataclass
class ClarificationQuestion:
    question_id: str
    prompt: str
    options: List[str]


@dataclass
class ArchitectureSpec:
    title: str
    decisions: Dict[str, str]
    concrete_examples: List[str] = field(default_factory=list)


class RefinementCoordinator:
    def evaluate_requirement_clarity(self, raw_requirement: str) -> Tuple[bool, List[ClarificationQuestion]]:
        """Detects underspecified requirements and generates clarifying questions."""
        questions = []
        lower = raw_requirement.lower()

        if "auth" in lower and not any(k in lower for k in ["jwt", "session", "oauth"]):
            questions.append(ClarificationQuestion(
                question_id="auth_type",
                prompt="Which authentication mechanism is required?",
                options=["JWT Bearer tokens", "Server-side cookie sessions", "OAuth2 / OIDC provider"],
            ))

        if "database" in lower or "store" in lower:
            if not any(k in lower for k in ["postgres", "sqlite", "mongo", "redis"]):
                questions.append(ClarificationQuestion(
                    question_id="storage_backend",
                    prompt="Which database storage backend should be configured?",
                    options=["PostgreSQL (SQLAlchemy)", "SQLite (Local embedded)", "Redis (In-memory)"],
                ))

        is_ambiguous = len(questions) > 0
        return is_ambiguous, questions

    def formulate_grounded_spec(self, title: str, user_answers: Dict[str, str]) -> ArchitectureSpec:
        """Compiles grounded architectural specification with concrete examples."""
        examples = [
            f"API Request Example: Authorization: {user_answers.get('auth_type', 'Bearer <token>')}",
            f"Storage Model: Dialect configured for {user_answers.get('storage_backend', 'PostgreSQL')}",
        ]
        return ArchitectureSpec(title=title, decisions=user_answers, concrete_examples=examples)

print(" RefinementCoordinator initialized.")

### 🧪 Step 3: Interactive Simulation
1. Evaluate an ambiguous prompt: "Build an auth service storing records in database".
2. Present clarifying questions.
3. Simulate user selecting options and generate grounded spec.

In [ ]:
# Step 3: Simulation
coordinator = RefinementCoordinator()

raw_prompt = "Build an auth service storing records in database"
is_ambiguous, questions = coordinator.evaluate_requirement_clarity(raw_prompt)

print("Is prompt ambiguous?", is_ambiguous)
print(f"Clarifying questions generated: {len(questions)}")
for q in questions:
    print(f"  [{q.question_id}]: {q.prompt}")
    for opt in q.options:
        print(f"      * {opt}")

# User answers
user_selections = {
    "auth_type": "JWT Bearer tokens",
    "storage_backend": "PostgreSQL (SQLAlchemy)",
}
grounded_spec = coordinator.formulate_grounded_spec("Authentication Service Spec", user_selections)
print("\nGrounded Specification Title:", grounded_spec.title)
print("Decisions:", grounded_spec.decisions)
print("Concrete Examples:", grounded_spec.concrete_examples)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Ambiguous requirements trigger at least 2 clarifying questions.
2. Grounded specification includes decisions and concrete examples.

In [ ]:
# Step 4: Verification Checks
assert is_ambiguous is True
assert len(questions) >= 2
q_ids = [q.question_id for q in questions]
assert "auth_type" in q_ids
assert "storage_backend" in q_ids

assert grounded_spec.decisions["auth_type"] == "JWT Bearer tokens"
assert len(grounded_spec.concrete_examples) == 2

print(" All verification assertions passed! Interview pattern verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: When requirements lack technical specifics, **interview the user**; do not guess or write speculative code.
- **Rule 2**: Provide multiple-choice options with architectural trade-offs.
- **Rule 3**: Formulate a grounded specification with concrete examples before implementation.